In [ ]:
Name: Sania
CMS ID: 023-24-0277

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix)
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    test_preds=model.predict(X_test)
    test_probs= model.predict_proba(X_test)[:,1]

    metrics={
        "Accuracy": accuracy_score(y_test, test_preds),
        "Precision": precision_score(y_test, test_preds),
        "Recall": recall_score(y_test, test_preds),
        "F1": f1_score(y_test, test_preds),
        "ROC AUC": roc_auc_score(y_test, test_probs),
        "PR AUC": average_precision_score(y_test, test_probs)        
    }
    print(confusion_matrix(y_test, test_preds))
    
    return pd.Series(metrics, name="Test Performance")


In [3]:
df=pd.read_csv("healthcare-dataset-stroke-data.csv")

In [4]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [5]:
print(df.shape)

(5110, 12)


In [6]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [7]:
df["gender"].value_counts()

gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64

In [8]:
df["stroke"].value_counts()

stroke
0    4861
1     249
Name: count, dtype: int64

In [9]:
df["stroke"].value_counts(normalize=True)*100

stroke
0    95.127202
1     4.872798
Name: proportion, dtype: float64

In [10]:
df=df.drop(columns=["id"])

In [11]:
df=df[df["gender"] !="Other"].copy()

In [12]:
X=pd.get_dummies(df,drop_first=True, dtype=int)
X=X.drop(columns=["stroke"])

In [13]:
X.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,1,0,1,0,0,1,1,0,0
1,61.0,0,0,202.21,NaN,0,1,0,0,1,0,0,0,1,0
2,80.0,0,1,105.92,32.5,1,1,0,1,0,0,0,0,1,0
3,49.0,0,0,171.23,34.4,0,1,0,1,0,0,1,0,0,1
4,79.0,1,0,174.12,24.0,0,1,0,0,1,0,0,0,1,0


In [14]:
y=df["stroke"]

In [15]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: stroke, dtype: int64

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)


In [17]:
mean=X_train["bmi"].mean()
print(mean)

28.881957186544344


In [18]:
X_train["bmi"]=X_train["bmi"].fillna(mean)
X_test["bmi"]=X_test["bmi"].fillna(mean)
print("Total null values in X train: ", X_train["bmi"].isnull().sum())
print("Total null values in X test: ",X_test["bmi"].isnull().sum())

Total null values in X train:  0
Total null values in X test:  0


In [19]:
X_train.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,48.0,0,0,69.21,33.1,0,1,0,1,0,0,1,0,1,0
3745,29.0,0,0,84.19,21.2,0,0,0,1,0,0,1,0,1,0
4184,35.0,0,0,119.40,22.9,0,1,0,1,0,0,0,0,1,0
3410,38.0,0,0,108.68,32.7,1,1,0,1,0,0,0,0,1,0
284,14.0,0,0,82.34,31.6,1,0,0,0,0,0,1,0,0,0


In [20]:
X_test.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
3667,28.0,1,0,83.66,36.4,0,1,0,0,0,0,0,0,1,0
2217,3.0,0,0,93.30,19.5,0,0,0,0,0,1,1,0,0,0
374,53.0,0,0,64.40,31.0,0,1,0,0,0,0,0,0,0,1
2392,35.0,0,0,69.22,42.8,1,1,0,1,0,0,0,0,1,0
299,58.0,0,0,94.30,29.1,0,1,0,1,0,0,0,0,0,0


In [21]:
X_test.shape

(1022, 15)

In [22]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler().fit(X_train)

scaler.set_output(transform="pandas")

X_train_s=scaler.transform(X_train)
X_test_s=scaler.transform(X_test)

In [23]:
X_train_s.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0.209397,-0.323411,-0.245319,-0.821221,0.545309,-0.839671,0.717905,-0.062692,0.870980,-0.440434,-0.396402,0.994875,-0.460478,1.299544,-0.421272
3745,-0.629845,-0.323411,-0.245319,-0.485884,-0.993125,-0.839671,-1.392942,-0.062692,0.870980,-0.440434,-0.396402,0.994875,-0.460478,1.299544,-0.421272
4184,-0.364822,-0.323411,-0.245319,0.302317,-0.773349,-0.839671,0.717905,-0.062692,0.870980,-0.440434,-0.396402,-1.005152,-0.460478,1.299544,-0.421272
3410,-0.232310,-0.323411,-0.245319,0.062342,0.493597,1.190942,0.717905,-0.062692,0.870980,-0.440434,-0.396402,-1.005152,-0.460478,1.299544,-0.421272
284,-1.292405,-0.323411,-0.245319,-0.527297,0.351389,1.190942,-1.392942,-0.062692,-1.148132,-0.440434,-0.396402,0.994875,-0.460478,-0.769500,-0.421272


In [24]:
print(y_train[y_train==1].value_counts())
print(y_test[y_test==1].value_counts())

stroke
1    199
Name: count, dtype: int64
stroke
1    50
Name: count, dtype: int64


**Part 0**     
a)
4.87% of data is postive and remaining is negative which is massive class imbalance. Stratify=y is used to keep the class imbalance ratio same in train and test set. If there are few stroke cases in the test set model will give fake high accuracy.

In [25]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline=DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train_s, y_train)

y_pred_base=baseline.predict(X_test_s)
base_acc=accuracy_score(y_test, y_pred_base)

print(confusion_matrix(y_test, baseline.predict(X_test)))

print(base_acc)

[[972   0]
 [ 50   0]]
0.9510763209393346


In [26]:
from sklearn.linear_model import LogisticRegression

SEED=42
lr=LogisticRegression(max_iter=1000, random_state=SEED)
results = evaluate_model(lr, X_train_s, y_train, X_test_s, y_test)
print(results)

[[972   0]
 [ 49   1]]
Accuracy     0.952055
Precision    1.000000
Recall       0.020000
F1           0.039216
ROC AUC      0.839403
PR AUC       0.267718
Name: Test Performance, dtype: float64


**Part 1**    
a) The accuracy improvement is negligible; both are essentially the same on accuracy.
With ~95% negatives, accuracy is misleading; a model can score ~95% by predicting almost all 0s.

b) A false negative means missing a real stroke case, dangerous for patient safety. Recall would be helpful as FN is more critical here.



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
depths=list(range(1,16))+[None]
train_auc=[]
cv_auc=[]

for d in depths:
    clf=DecisionTreeClassifier(max_depth=d, random_state=SEED)
    scores=cross_val_score(clf, X_train_s, y_train, scoring="roc_auc", cv=cv)
    cv_auc.append(scores.mean())

    clf.fit(X_train_s, y_train)
    train_auc.append(roc_auc_score(y_train, clf.predict_proba(X_train_s)[:,1]))

results=pd.DataFrame({"max_depth":depths, "train_auc":train_auc, "cv_auc":cv_auc})
print(results)
print("Best depth:", results.loc[results["cv_auc"].idxmax(), "max_depth"])

In [ ]:
plt.plot([str(d) for d in depths], train_auc, label="Train AUC")
plt.plot([str(d) for d in depths], cv_auc, label="CV AUC")
plt.xlabel("Max depth")
plt.ylabel("ROC AUC")
plt.title("Validation curve")
plt.legend()
plt.xticks(rotation=45)
plt.show()

The shallow trees have low train and CV AUC, so they underfit. Deep trees get very high train AUC but lower CV AUC, which shows overfitting. I would choose the depth with the highest CV AUC because it performs best on unseen folds.

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes=plt.subplots(1,2,figsize=(12,4))
for depth, ax in zip([1,None], axes):
    tree=DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    sizes, train_scores, valid_scores=learning_curve(
        tree, X_train_s, y_train, cv=cv, scoring="roc_auc",
        train_sizes=np.linspace(0.2,1.0,5), n_jobs=-1
    )
    ax.plot(sizes, train_scores.mean(axis=1), label="Train")
    ax.plot(sizes, valid_scores.mean(axis=1), label="CV")
    ax.set_title("Depth = "+str(depth))
    ax.set_xlabel("Training rows")
    ax.set_ylabel("ROC AUC")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
rng=np.random.default_rng(SEED)
variance_rows=[]
for depth in [1,3,None]:
    predictions=[]
    aucs=[]
    for i in range(40):
        idx=rng.choice(len(X_train_s), len(X_train_s), replace=True)
        tree=DecisionTreeClassifier(max_depth=depth, random_state=SEED+i)
        tree.fit(X_train_s.iloc[idx], y_train.iloc[idx])
        prob=tree.predict_proba(X_test_s)[:,1]
        predictions.append(prob)
        aucs.append(roc_auc_score(y_test, prob))
    predictions=np.array(predictions)
    avg_prob=predictions.mean(axis=0)
    variance_rows.append({
        "Depth":str(depth),
        "Mean prediction variance":predictions.var(axis=0).mean(),
        "Mean tree AUC":np.mean(aucs),
        "AUC averaged prediction":roc_auc_score(y_test, avg_prob)
    })
variance_table=pd.DataFrame(variance_rows)
print(variance_table)

The variance generally increases as the tree becomes deeper because deep trees react more to changes in training rows. The averaged prediction is usually more stable and can have better AUC than individual trees. Random Forest uses the same idea by averaging many trees. Depth 1 has high bias, depth 3 is a simpler middle ground, and an unlimited tree has high variance. Adding data can help high variance, but it may not fix a model that is too simple.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.model_selection import cross_val_predict

models={
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED)
}
probabilities={}
for name, model in models.items():
    if name=="Logistic Regression":
        model.fit(X_train_s, y_train)
        probabilities[name]=model.predict_proba(X_test_s)[:,1]
    else:
        model.fit(X_train, y_train)
        probabilities[name]=model.predict_proba(X_test)[:,1]

plt.figure(figsize=(6,5))
for name, prob in probabilities.items():
    fpr,tpr,_=roc_curve(y_test,prob)
    auc=roc_auc_score(y_test,prob)
    plt.plot(fpr,tpr,label=name+" AUC="+str(round(auc,3)))
plt.plot([0,1],[0,1],"--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.show()

plt.figure(figsize=(6,5))
for name, prob in probabilities.items():
    precision,recall,_=precision_recall_curve(y_test,prob)
    ap=average_precision_score(y_test,prob)
    plt.plot(recall,precision,label=name+" AP="+str(round(ap,3)))
plt.axhline(y_train.mean(), linestyle="--", label="No skill")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves")
plt.legend()
plt.show()

ROC AUC measures how well the model ranks stroke patients above non-stroke patients. It does not depend on using 0.5 as the threshold. A model can have good AUC but low recall at 0.5 because the predicted probabilities for stroke patients may still be below that cutoff. The Precision-Recall curve is more useful here because stroke is rare; its no-skill line is the small stroke rate, so precision above that line is useful.

The lower threshold catches more stroke cases, but it also flags more patients who did not have a stroke. This means more follow-up tests and extra workload for the hospital. For screening, this may be acceptable because missing a stroke can be more serious than checking an extra patient.

Resampling mainly changes the class balance seen during training. Recall often increases because the model pays more attention to stroke cases, while precision can decrease because more non-stroke patients are flagged. ROC AUC may change less because it measures ranking rather than one fixed cutoff. Logistic Regression and Random Forest can react differently because they learn patterns in different ways. I would choose the method that gives useful recall without creating too many false alarms.

In [ ]:
X_all=pd.get_dummies(df.drop(columns=["stroke"]),drop_first=True,dtype=int)
y_all=df["stroke"]
X_all["bmi"]=X_all["bmi"].fillna(X_all["bmi"].median())
X_leak,y_leak=SMOTE(random_state=SEED).fit_resample(X_all,y_all)
X_leak_train,X_leak_test,y_leak_train,y_leak_test=train_test_split(
    X_leak,y_leak,test_size=0.2,random_state=SEED,stratify=y_leak
)
wrong_model=RandomForestClassifier(n_estimators=200,min_samples_leaf=5,random_state=SEED)
wrong_model.fit(X_leak_train,y_leak_train)
wrong_prob=wrong_model.predict_proba(X_leak_test)[:,1]
wrong_pred=wrong_model.predict(X_leak_test)

correct_smote=comparison_table[
    (comparison_table["Model"]=="Random Forest") &
    (comparison_table["Method"]=="SMOTE")
].iloc[0]
wrong_table=pd.DataFrame([
    {"Result":"Correct SMOTE after split","Precision":correct_smote["Precision"],
     "Recall":correct_smote["Recall"],"F1":correct_smote["F1"],
     "ROC AUC":correct_smote["ROC AUC"],"PR AUC":correct_smote["PR AUC"]},
    {"Result":"Wrong SMOTE before split","Precision":precision_score(y_leak_test,wrong_pred),
     "Recall":recall_score(y_leak_test,wrong_pred),"F1":f1_score(y_leak_test,wrong_pred),
     "ROC AUC":roc_auc_score(y_leak_test,wrong_prob),
     "PR AUC":average_precision_score(y_leak_test,wrong_prob)}
])
print(wrong_table)

The wrong-order result can look better because SMOTE creates synthetic patients before the split. Similar or related synthetic examples can then appear in both training and test sets. The test set is no longer independent, so these scores are not trustworthy. SMOTE must be applied only after splitting, and only to the training data.

SMOTE may amplify label noise because it treats every minority label as correct. If a wrong label is included among the stroke examples, SMOTE can create more synthetic examples around that incorrect point. I compare PR AUC and ROC AUC to see whether this helped or hurt each model.

In [ ]:
final_model=LogisticRegression(max_iter=1000,random_state=SEED)
final_model.fit(X_train_s,y_train)
final_prob=final_model.predict_proba(X_test_s)[:,1]
final_rows=[]
for label,threshold in [("Default 0.5",0.5),("Chosen threshold",chosen_threshold)]:
    pred=(final_prob>=threshold).astype(int)
    cm=confusion_matrix(y_test,pred)
    final_rows.append({
        "Threshold":label,"Precision":precision_score(y_test,pred,zero_division=0),
        "Recall":recall_score(y_test,pred,zero_division=0),
        "F1":f1_score(y_test,pred,zero_division=0),
        "False alarms":cm[0,1],"Missed strokes":cm[1,0]
    })
final_table=pd.DataFrame(final_rows)
print(final_table)

For the hospital, I would use the Logistic Regression model with the threshold selected from training cross-validation. Its test recall is [use the value above] and its precision is [use the value above]. The cost is that more patients without stroke may be flagged, but fewer real stroke cases are missed. I avoided leakage by fitting preprocessing and choosing the threshold using training data only. One limitation is that the test set contains few stroke patients, so the measured recall can change noticeably with a small number of cases.

The most surprising result was that a model can have high accuracy and still miss many stroke patients. With one more hour, I would try more threshold values and compare the number of missed strokes with the number of false alarms.